<a href="https://colab.research.google.com/github/NeDarbandsari/Facial-Emotion-Detection/blob/main/Final_Edited_Emotion_Detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from IPython import display
!pip install deepface
display.clear_output()

from deepface import DeepFace
import cv2
import os
from google.colab import drive

import pandas as pd
import json

from itertools import groupby

25-12-01 10:29:54 - Directory /root/.deepface has been created
25-12-01 10:29:54 - Directory /root/.deepface/weights has been created


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
drive.mount("/content/drive", force_remount=True)

ZIP_PATH = "/content/drive/MyDrive/Recordings.zip"
ROOT_DIR = "/content/drive/MyDrive/Recordings"
OUTPUT_CSV = "/content/drive/MyDrive/emotion_results.csv"
ALLOWED_EXTS = (".mp4", ".mov", ".avi", ".mkv", ".m4v")


if not os.path.exists(ROOT_DIR):
    print("Extracting dataset, please wait...")
    import zipfile
    with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
        zip_ref.extractall(ROOT_DIR)
    print("Extraction complete:", ROOT_DIR)
else:
    print("Dataset folder already exists:", ROOT_DIR)

Mounted at /content/drive
Dataset folder already exists: /content/drive/MyDrive/Recordings


In [4]:
def iter_videos(root_dir, allowed_exts=ALLOWED_EXTS):
    for dirpath, dirnames, filenames in os.walk(root_dir):
        dirnames.sort()
        for fn in sorted(filenames):
            if fn.lower().endswith(allowed_exts):
                yield dirpath, fn, os.path.join(dirpath, fn)

all_videos = list(iter_videos(ROOT_DIR))
print(f"Found {len(all_videos)} video(s). Example:", all_videos[0])

Found 1002 video(s). Example: ('/content/drive/MyDrive/Recordings/01', '01-01-01-01.mp4', '/content/drive/MyDrive/Recordings/01/01-01-01-01.mp4')


In [5]:
def video_emotion_detection(video, backends, alignment_modes, model,video_path):
  emotion_model = DeepFace.build_model(model)
  DeepFace.custom_models = {model: emotion_model}
  print("processing ", video)

  cap = cv2.VideoCapture(os.path.join(video_path, video))

  fps = cap.get(cv2.CAP_PROP_FPS) or 25.0
  frame_interval = int(fps)

  prev_emotion = None
  start_time = None
  frame_count = 0
  result = {}
  result['file'] = video
  result['model'] = model
  result['backends'] = backends

  emotion=[]

  while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

      # Process only every frame_interval-th frame
    if frame_count % frame_interval == 0:
        process = {}
        current_time = frame_count / fps  # current time in seconds
        try:
            # Analyze the current frame for emotion
            analysis = DeepFace.analyze(frame, actions=['emotion'], enforce_detection=False, detector_backend = backends)

            # If the result is a list (multiple faces), take the first one
            if isinstance(analysis, list):
                analysis = analysis[0]
            curr_emotion = analysis['dominant_emotion']

            # Initialize on the first processed frame
            if prev_emotion is None:
                prev_emotion = curr_emotion
                start_time = current_time
                print(f"Starting with emotion '{curr_emotion}' at {current_time:.1f}s")
                process['starting emotion'] = curr_emotion
                process['time'] = current_time
            # When the emotion changes, print the duration and update the tracker
            elif curr_emotion != prev_emotion:
                duration = current_time - start_time
                print(f"Emotion '{prev_emotion}' lasted for {duration:.1f} seconds")
                process['prev_emotion'] = prev_emotion
                process['duration'] = duration
                print(f"At {current_time:.1f}s, dominant emotion changed to '{curr_emotion}'")
                process['curr_emotion'] = curr_emotion
                process['current_time'] =current_time
                prev_emotion = curr_emotion
                start_time = current_time
            if len(process) != 0:
              emotion.append(process)
        except Exception as e:
            print(f"Error analyzing frame {frame_count}: {e}")


    frame_count += 1
  cap.release()
  # Print the duration for the final emotion
  total_time = frame_count / fps
  if prev_emotion is not None and start_time is not None:
    final_duration = total_time - start_time
    print(f"Final emotion '{prev_emotion}' lasted for {final_duration:.1f} seconds")
  print("Analysis complete.")
  result['emotions'] = emotion
  return result

In [6]:
backends = [
  'opencv',
  'ssd',
  'dlib',
  'mtcnn',
  'fastmtcnn',
  'retinaface',
  'mediapipe',
  'yolov8',
  'yolov11s',
  'yolov11n',
  'yolov11m',
  'yunet',
  'centerface',
]

alignment_modes = [True, False]

supported_models = [
    "VGG-Face",
    "Facenet",
    "Facenet512",
    "OpenFace",
    "DeepFace",
    "DeepID",
    "ArcFace",
    "Dlib",
    "SFace",
]

In [7]:
chosen_backend   = backends[3]          # mtcnn
chosen_alignment = alignment_modes[0]   # True
chosen_model     = supported_models[3]  # OpenFace


def result_to_rows(r):
    """Flatten one video result dict into per-segment CSV rows."""
    base = {
        "folder": r.get("folder"),
        "file":   r.get("file"),
        "path":   r.get("path"),
        "model":  r.get("model"),
        "backend":r.get("backends"),
    }
    rows = []
    for emo in r.get("emotions", []):
        row = base.copy()
        row.update(emo)
        rows.append(row)
    return rows

def append_rows(rows, csv_path):
    """Append rows to CSV safely."""
    if not rows:
        return 0
    df = pd.DataFrame(rows)
    header_needed = (not os.path.exists(csv_path)) or os.stat(csv_path).st_size == 0
    df.to_csv(csv_path, mode="a", header=header_needed, index=False)
    return len(df)


# 1) Load existing CSV and mark already-processed videos
if os.path.exists(OUTPUT_CSV):
    try:
        existing = pd.read_csv(OUTPUT_CSV, usecols=["path"])
        processed_paths = set(existing["path"].astype(str))
        print(f"Found {len(processed_paths)} already processed videos in CSV.")
    except Exception:
        processed_paths = set()
        print("Could not read previous CSV — starting fresh.")
else:
    processed_paths = set()
    print("No existing CSV found. Starting fresh.")


# 2) Group videos by folder and process only new ones
all_videos_sorted = sorted(all_videos, key=lambda t: (t[0], t[1]))
total_appended = 0
folders_done = 0

try:
    for folder_path, group in groupby(all_videos_sorted, key=lambda t: t[0]):
        group = list(group)
        print(f"\n=== Folder: {folder_path} ===")

        # filter only new videos
        new_files = [(p, f, fp) for (p, f, fp) in group if fp not in processed_paths]
        if not new_files:
            print("All videos in this folder already processed.")
            folders_done += 1
            continue

        folder_rows = []
        for _, file_name, file_path in new_files:
            print(">>>", file_path)
            out = video_emotion_detection(
                file_name, chosen_backend, chosen_alignment, chosen_model, folder_path + os.sep
            )
            out["folder"] = folder_path
            out["path"] = file_path
            folder_rows.extend(result_to_rows(out))
            processed_paths.add(file_path)  # mark done in memory

        added = append_rows(folder_rows, OUTPUT_CSV)
        total_appended += added

        print(f"Saved {added} new rows from folder: {folder_path}")
        folders_done += 1

        try:
            peek = pd.read_csv(OUTPUT_CSV).tail(3)
            display(peek)
        except Exception:
            pass

except KeyboardInterrupt:
    print("\nStopped manually — progress saved so far.")

finally:
    if os.path.exists(OUTPUT_CSV):
        total_rows = pd.read_csv(OUTPUT_CSV).shape[0]
        print(f"\nResume-ready CSV updated: {OUTPUT_CSV}")
        print(f"Total processed folders: {folders_done} | New rows added: {total_appended} | Total rows in CSV: {total_rows}")
    else:
        print("\nNo CSV file was written yet.")


Found 1002 already processed videos in CSV.

=== Folder: /content/drive/MyDrive/Recordings/01 ===
All videos in this folder already processed.

=== Folder: /content/drive/MyDrive/Recordings/02 ===
All videos in this folder already processed.

=== Folder: /content/drive/MyDrive/Recordings/03 ===
All videos in this folder already processed.

=== Folder: /content/drive/MyDrive/Recordings/04 ===
All videos in this folder already processed.

=== Folder: /content/drive/MyDrive/Recordings/05 ===
All videos in this folder already processed.

=== Folder: /content/drive/MyDrive/Recordings/06 ===
All videos in this folder already processed.

=== Folder: /content/drive/MyDrive/Recordings/07 ===
All videos in this folder already processed.

=== Folder: /content/drive/MyDrive/Recordings/08 ===
All videos in this folder already processed.

=== Folder: /content/drive/MyDrive/Recordings/09 ===
All videos in this folder already processed.

=== Folder: /content/drive/MyDrive/Recordings/10 ===
All videos 

In [8]:
# A separate CSV for evaluating multiple backend/model combinations
OUTPUT_CSV_COMBOS = "/content/drive/MyDrive/emotion_results_combos.csv"

# Best combinations for facial emotion recognition
BEST_COMBINATIONS = [
    ("retinaface", "Facenet512"),
    ("retinaface", "ArcFace"),

    ("mtcnn", "Facenet512"),
   # ("mtcnn", "OpenFace"),

   # ("mediapipe", "Facenet512"),
]

In [ ]:
import os
import pandas as pd

print("Running backend/model combination evaluation (safe resume)...")

# ---------- 1) Load already-processed combos from CSV ----------
if os.path.exists(OUTPUT_CSV_COMBOS):
    prev_df = pd.read_csv(OUTPUT_CSV_COMBOS)

    if {"path", "backend", "model"}.issubset(prev_df.columns):
        processed_combos = set(
            zip(
                prev_df["path"].astype(str),
                prev_df["backend"].astype(str),
                prev_df["model"].astype(str),
            )
        )
        print(f"Loaded {len(prev_df)} existing rows from {OUTPUT_CSV_COMBOS}")
        print(f"{len(processed_combos)} unique (path, backend, model) combos already processed.")
    else:
        processed_combos = set()
        print(f"{OUTPUT_CSV_COMBOS} exists but is missing required columns; starting fresh.")
else:
    processed_combos = set()
    print(f"No existing {OUTPUT_CSV_COMBOS} found. Starting fresh.")

# ---------- 2) Safe / resumable combo loop ----------
total_appended_combos = 0
folders_done = 0

for root, dirs, files in os.walk(ROOT_DIR):
    video_files = [f for f in files if f.lower().endswith(ALLOWED_EXTS)]
    if not video_files:
        continue

    print(f"\n Folder: {root}")

    # rows for this folder only
    folder_rows_combos = []

    try:
        for file_name in video_files:
            file_path = os.path.join(root, file_name)
            print(f"Processing: {file_path}")

            for backend_name, model_name in BEST_COMBINATIONS:
                combo_key = (file_path, backend_name, model_name)

                # skip if this combo has already been written before
                if combo_key in processed_combos:
                    # Uncomment for debugging:
                    # print(f"   ⏭ Already done: {combo_key}")
                    continue

                print(f"   → Combo: backend={backend_name}, model={model_name}")

                try:
                    # match your existing function signature: positional args
                    out_combo = video_emotion_detection(
                        file_name,        # video file name
                        backend_name,     # detector backend
                        chosen_alignment, # alignment
                        model_name,       # model
                        root + os.sep     # folder path
                    )
                except Exception as e:
                    # e.g. mediapipe not installed, etc.
                    print(f"Skipping {combo_key} due to error: {e}")
                    # do NOT add to processed_combos so you can fix & retry later
                    continue

                # add metadata
                out_combo["folder"] = root
                out_combo["path"] = file_path
                out_combo["backend"] = backend_name
                out_combo["model"] = model_name

                # convert to CSV rows (same helper as your original code)
                rows = result_to_rows(out_combo)
                folder_rows_combos.extend(rows)

                # mark as processed in-memory so this run doesn't repeat it
                processed_combos.add(combo_key)

    except KeyboardInterrupt:
        # if you manually stop in the middle of this folder, still save what we have
        if folder_rows_combos:
            added = append_rows(folder_rows_combos, OUTPUT_CSV_COMBOS)
            total_appended_combos += added
            print(f"\n⏸ KeyboardInterrupt – saved {added} rows from folder: {root} before stopping.")
        raise  # re-raise so the interrupt behaves normally

    # normal per-folder save
    if folder_rows_combos:
        added = append_rows(folder_rows_combos, OUTPUT_CSV_COMBOS)
        total_appended_combos += added
        print(f"   Saved {added} new rows from folder: {root}")

    folders_done += 1

print(f"\n✔ Finished combo evaluation.")
print(f"Total new rows added in this run: {total_appended_combos}")
print(f"Folders visited: {folders_done}")

Running backend/model combination evaluation (safe resume)...
Loaded 671 existing rows from /content/drive/MyDrive/emotion_results_combos.csv
270 unique (path, backend, model) combos already processed.

 Folder: /content/drive/MyDrive/Recordings/25
Processing: /content/drive/MyDrive/Recordings/25/03-01-25-01.mp4
Processing: /content/drive/MyDrive/Recordings/25/02-02-25-01.mp4
Processing: /content/drive/MyDrive/Recordings/25/01-02-25-01.mp4
Processing: /content/drive/MyDrive/Recordings/25/04-01-25-01.mp4
Processing: /content/drive/MyDrive/Recordings/25/03-02-25-01.mp4
Processing: /content/drive/MyDrive/Recordings/25/02-01-25-01.mp4
Processing: /content/drive/MyDrive/Recordings/25/01-01-25-01.mp4
Processing: /content/drive/MyDrive/Recordings/25/04-02-25-01.mp4
Processing: /content/drive/MyDrive/Recordings/25/05-01-25-01.mp4
Processing: /content/drive/MyDrive/Recordings/25/05-02-25-01.mp4

 Folder: /content/drive/MyDrive/Recordings/22
Processing: /content/drive/MyDrive/Recordings/22/03-01

Downloading...
From: https://github.com/serengil/deepface_models/releases/download/v1.0/facial_expression_model_weights.h5
To: /root/.deepface/weights/facial_expression_model_weights.h5
100%|██████████| 5.98M/5.98M [00:00<00:00, 118MB/s]


Starting with emotion 'neutral' at 0.0s
Emotion 'neutral' lasted for 1.0 seconds
At 1.0s, dominant emotion changed to 'fear'
Emotion 'fear' lasted for 1.0 seconds
At 2.0s, dominant emotion changed to 'angry'
Emotion 'angry' lasted for 1.0 seconds
At 3.0s, dominant emotion changed to 'fear'
Final emotion 'fear' lasted for 1.7 seconds
Analysis complete.
   → Combo: backend=retinaface, model=ArcFace
25-12-01 10:30:48 - 🔗 arcface_weights.h5 will be downloaded from https://github.com/serengil/deepface_models/releases/download/v1.0/arcface_weights.h5 to /root/.deepface/weights/arcface_weights.h5...


Downloading...
From: https://github.com/serengil/deepface_models/releases/download/v1.0/arcface_weights.h5
To: /root/.deepface/weights/arcface_weights.h5
100%|██████████| 137M/137M [00:00<00:00, 345MB/s]


processing  02-01-47-01.mp4
Starting with emotion 'neutral' at 0.0s
Emotion 'neutral' lasted for 1.0 seconds
At 1.0s, dominant emotion changed to 'fear'
Emotion 'fear' lasted for 1.0 seconds
At 2.0s, dominant emotion changed to 'angry'
Emotion 'angry' lasted for 1.0 seconds
At 3.0s, dominant emotion changed to 'fear'
Final emotion 'fear' lasted for 1.7 seconds
Analysis complete.
   → Combo: backend=mtcnn, model=Facenet512
processing  02-01-47-01.mp4
Starting with emotion 'neutral' at 0.0s
Emotion 'neutral' lasted for 1.0 seconds
At 1.0s, dominant emotion changed to 'fear'
Emotion 'fear' lasted for 2.0 seconds
At 3.0s, dominant emotion changed to 'angry'
Emotion 'angry' lasted for 1.0 seconds
At 4.0s, dominant emotion changed to 'surprise'
Final emotion 'surprise' lasted for 0.7 seconds
Analysis complete.
Processing: /content/drive/MyDrive/Recordings/47/01-01-47-01.mp4
   → Combo: backend=retinaface, model=Facenet512
processing  01-01-47-01.mp4
Starting with emotion 'happy' at 0.0s
Fina